# 📍 Phase 2 — Géo-Enrichissement (OpenStreetMap)
## Présentation — ~2 minutes
---

## Le problème

Les annonces contiennent juste le nom du quartier en texte : "Tevragh Zeina", "Arafat", etc. Mais un modèle de ML a besoin de **chiffres**, pas de texte.

**Question :** Comment transformer un nom de quartier en features exploitables par le modèle ?

**Réponse :** On utilise OpenStreetMap pour enrichir chaque annonce avec des données géographiques.

## Ce qu'on a fait — 3 étapes

### 1. Géocodage avec Nominatim

On convertit chaque quartier en coordonnées GPS via l'API gratuite **Nominatim** d'OpenStreetMap.

| Quartier | Latitude | Longitude | Caractère |
|----------|----------|-----------|-----------|
| Tevragh Zeina | 18.1036 | -15.9785 | Quartier huppé, ambassades |
| Ksar | 18.0866 | -15.9750 | Centre historique |
| Arafat | 18.0550 | -15.9610 | Populaire, dense |
| Dar Naim | 18.1200 | -15.9450 | Résidentiel, en expansion |
| Toujounine | 18.0680 | -15.9350 | Périphérie, récent |
| Sebkha | 18.0730 | -15.9870 | Commercial, marchés |
| Riyadh | 18.0850 | -15.9550 | Résidentiel moyen |
| Teyarett | 18.0950 | -15.9700 | Centre, mixte |

On utilise des coordonnées de référence fiables car Nominatim peut être imprécis pour la Mauritanie. On vérifie avec l'API et on prend un fallback si nécessaire.

### 2. Calcul des distances avec Geopy

Pour chaque annonce, on calcule la distance géodésique (en km) vers 5 points de référence de Nouakchott :

| Distance | Point de référence | Pourquoi c'est utile |
|----------|-------------------|---------------------|
| `dist_centre_ville_km` | Ksar (centre historique) | Proximité au centre = plus cher |
| `dist_aeroport_km` | Aéroport Oumtounsy | Accessibilité |
| `dist_plage_km` | Plage de Nouakchott | Bord de mer = premium |
| `dist_grand_marche_km` | Grand Marché | Zone commerciale |
| `dist_port_km` | Port de l'Amitié | Zone industrielle |

On utilise la formule **géodésique** (pas euclidienne) pour avoir des distances réelles sur la surface de la Terre.

### 3. Points d'Intérêt (POI) via Overpass API

L'API **Overpass** interroge la base de données OpenStreetMap pour compter les POI dans un rayon de 1 km autour de chaque quartier :

| Feature | Ce qu'on compte | Proxy de |
|---------|----------------|----------|
| `nb_ecoles_1km` | Écoles | Quartier familial |
| `nb_mosquees_1km` | Mosquées / lieux de culte | Densité urbaine |
| `nb_commerces_1km` | Tous les commerces | Attractivité commerciale |
| `nb_hopitaux_1km` | Hôpitaux, cliniques | Qualité de vie |
| `nb_restaurants_1km` | Restaurants | Standing du quartier |
| `nb_banks_1km` | Banques | Zone d'affaires |
| `nb_total_pois_1km` | Total de tous les POI | Proxy d'urbanisation |

On requête **par quartier** (pas par annonce) pour respecter le rate-limit de l'API et ne pas surcharger les serveurs.

## Résultat

**13 nouvelles features géographiques** ajoutées au dataset :
- 2 coordonnées (latitude, longitude)
- 5 distances (centre, aéroport, plage, marché, port)
- 6 comptages de POI + 1 total

**Visualisation :** On a aussi généré une carte Folium interactive avec les annonces colorées par quartier et une heatmap des prix.

## Limites honnêtes

1. **Précision au niveau quartier, pas de l'adresse** — toutes les annonces d'Arafat ont les mêmes coordonnées
2. **OpenStreetMap est incomplet pour Nouakchott** — peu de POI référencés par rapport à une ville européenne
3. **Distances à vol d'oiseau** — pas en distance routière (pas de données routières fiables pour Nouakchott)

## Impact sur la modélisation

Ces features géo n'ont **pas amélioré** le score Kaggle car on n'a que 8 quartiers — le target encoding capture déjà toute l'information géographique de manière plus directe. Mais elles sont utiles pour **l'interprétabilité** et pour **l'application web** (carte interactive).

---
*→ Transition : le dataset enrichi alimente la Phase 3 (EDA) puis la Phase 4 (Modélisation)*